In [19]:
import numpy as np
from scipy.io import wavfile

fs, data = wavfile.read('POCSAG_IQ.wav')
i_float = data[:, 0].astype(np.float64) - 128.0
q_float = data[:, 1].astype(np.float64) - 128.0
iq = i_float + 1j * q_float

# phase chnage detection 
freq = np.angle(iq[1:] * np.conj(iq[:-1]))
freq_centered = freq - np.median(freq)

def codeword_ok(cw):
    reg = cw >> 1                           # removin g last party bit
    for i in range(30, 9, -1):              # xor l ong division
        if reg & (1 << i):
            reg ^= 0x769 << (i - 10)
    return reg == 0 and bin(cw).count('1') % 2 == 0


SYNC = format(0x7CD215D8, '032b')           
IDLE = 0x7A89C197                          

def payload_to_text(payloads):
    s = ''.join(format(p, '020b') for p in payloads)
    text = ''
    for k in range(0, len(s) - 6, 7):
        c = int(s[k:k+7][::-1], 2)          
        if 32 <= c < 127:
            text += chr(c)
    return text

def decode(bit_string):
    messages, current = [], None
    pos = bit_string.find(SYNC)
    while pos != -1:
        for k in range(16):                 
            a = pos + 32 + 32 * k
            if a + 32 > len(bit_string):
                break
            cw = int(bit_string[a:a+32], 2)
            if cw == IDLE:
                current = None
            elif not codeword_ok(cw):
                continue
            elif cw >> 31 == 0:             # label 0 = address
                addr = (((cw >> 13) & 0x3FFFF) << 3) | (k // 2)
                current = {'addr': addr, 'payloads': []}
                messages.append(current)
            elif current is not None:       # label 1 = message
                current['payloads'].append((cw >> 11) & 0xFFFFF)
        pos = bit_string.find(SYNC, pos + 32)
    return [(m['addr'], payload_to_text(m['payloads'])) for m in messages]



def find_messages():
    for baud in (512, 1200, 2400):
        sps = fs / baud                     # samples per sybmol or bits we can sya 
        k = max(1, int(sps / 2))
        smooth = np.convolve(freq_centered, np.ones(k) / k, mode='same')
        for offset in np.arange(0, sps, max(1.0, sps / 16)):
            idx = (offset + np.arange(int((len(smooth) - offset) / sps)) * sps).astype(int)
            raw_bits = (smooth[idx] < 0).astype(int)
            for invert in (False, True):
                b = 1 - raw_bits if invert else raw_bits
                msgs = decode(''.join(map(str, b)))
                if msgs:
                    print(f'baud={baud}, offset={offset:.1f}, invert={invert}')
                    return msgs
    return []

for addr, text in find_messages():
    print(f'Address {addr}: "{text}"')

baud=512, offset=125.0, invert=True
Address 1234567: "THIS IS A TEST PERIODIC PAGE SEQUENTIAL NUMBER  2379"
